# RuO2 dissolution roughness models in DWBA

This notebook reproduces the two roughness families in `RuO2_roughness_models.ipynb` with atomic DWBA:

1. signed Poisson dissolution of the RuO2 surface;
2. deterministic removal from an explicit six-layer surface `UnitCell`.

For every model it evaluates the specular `(0, 0, L)` rod and the non-specular `(0, 1, L)` rod. The latter is calculated with $\alpha_i=1.5\alpha_c$, where $\alpha_c$ is obtained from the TiO2 substrate dispersion. Specular solid curves use the total DWBA field expressed in electron-equivalent structure-factor units; non-specular solid curves use the coherent contrast factor. Dashed curves are the corresponding kinematical amplitudes.

In [ ]:
%matplotlib widget
from pathlib import Path
import copy
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np


def find_repository_root():
    """Find the checkout containing the current example notebook."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "orgui" / "datautils").is_dir():
            return candidate
    return None


repository_root = find_repository_root()
if repository_root is not None and "orgui" not in sys.modules:
    sys.path.insert(0, str(repository_root))

from orgui.datautils.xrayutils import CTRcalc, CTRuc
from orgui.datautils.xrayutils.CTRoptics import homogeneous_bulk_profile


def find_example_file(filename):
    """Find an example file from the checkout or notebook directory."""
    candidates = (Path(filename), Path("examples/CTR") / filename)
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(filename)


if not hasattr(CTRcalc.SXRDCrystal, "dwba"):
    raise RuntimeError("The loaded orGUI build does not provide DWBA.")
if not hasattr(CTRuc._CTRcalc_cpp, "unitcell_F_DWBA_records"):
    raise RuntimeError(
        "Rebuild the native extension before running this notebook."
    )

## Shared model and rod evaluation

The kinematical calculations retain the serialized model attenuation. DWBA does not reuse that empirical factor: its complex internal wavevectors already contain absorption. Off specular, `F_contrast` is directly comparable to `SXRDCrystal.F`. Specularly, `F_contrast` is only the correction, so the notebook converts `total_amplitude = r0 + scattered_amplitude` back to equivalent electron units before comparing it with the kinematical structure factor.

In [ ]:
model_path = find_example_file("RuO2_TiO2_Poisson_etching.xtal")
model_template = CTRcalc.SXRDCrystal.fromFile(model_path)

delta_bulk = float(
    homogeneous_bulk_profile(model_template.uc_bulk).values[0, 1]
)
alpha_c = float(np.sqrt(2.0 * delta_bulk))
alpha_i_fixed = 1.5 * alpha_c

L_specular = np.linspace(0.002, 7.0, 2001)
L_rod = np.linspace(0.06, 6.0, 2001)
z = np.linspace(-10.0, 60.0, 2600)
CLASSICAL_ELECTRON_RADIUS_ANGSTROM = 2.8179403262e-5


def field_to_structure_factor(amplitude, prepared):
    """Express a field amplitude in equivalent electron units."""
    return (
        amplitude * prepared.k0 * np.sin(prepared.alpha_f)
        * prepared.reference_area
        / (2j * np.pi * CLASSICAL_ELECTRON_RADIUS_ANGSTROM)
    )


def evaluate_model(model):
    """Evaluate matched kinematical and DWBA rods for one model."""
    h_specular = np.zeros_like(L_specular)
    k_specular = np.zeros_like(L_specular)
    F_kinematic_specular = model.F(
        h_specular, k_specular, L_specular
    )
    model.dwba.set_ctr_geometry(
        equal_angles=True, rods=[(0.0, 0.0)]
    )
    specular = model.dwba.evaluate(
        h_specular, k_specular, L_specular
    )

    h_rod = np.zeros_like(L_rod)
    k_rod = np.ones_like(L_rod)
    F_kinematic_rod = model.F(h_rod, k_rod, L_rod)
    model.dwba.set_ctr_geometry(
        alpha_i=alpha_i_fixed, rods=[(0.0, 1.0)]
    )
    non_specular = model.dwba.evaluate(h_rod, k_rod, L_rod)

    assert np.all(specular.prepared.is_specular)
    assert not np.any(non_specular.prepared.is_specular)
    np.testing.assert_allclose(non_specular.F_reference, 0.0)
    return {
        "F_kinematic_specular": F_kinematic_specular,
        "F_dwba_specular": field_to_structure_factor(
            specular.total_amplitude, specular.prepared
        ),
        "reflectivity": specular.reflectivity,
        "F_kinematic_rod": F_kinematic_rod,
        "F_dwba_rod": non_specular.F_contrast,
        "density": np.abs(model.zDensity_G(z, 0, 0)),
    }


print(f"energy: {model_template.uc_bulk._E * 1e-3:g} keV")
print(f"alpha_c: {np.rad2deg(alpha_c):.5f} deg")
print(f"fixed alpha_i: {np.rad2deg(alpha_i_fixed):.5f} deg")

## Signed Poisson dissolution

As in the original notebook, `mean_change = -N` and `offset = 0`. Increasing $N$ lowers the mean surface and broadens it. Each generated signed Poisson record remains coherent inside its owning surface contribution.

In [ ]:
dissolved_layers = np.arange(0, 7, dtype=float)
poisson_results = []

for dissolved in dissolved_layers:
    model = copy.deepcopy(model_template)
    surface = model["RuO2surface"]
    surface.profile = CTRcalc.PoissonProfile(
        mean_change=-dissolved, alpha=surface.basis[1], offset=0.0
    )
    surface.basis[:] = (
        surface.profile.mean_change, surface.profile.alpha,
        surface.profile.offset,
    )
    surface.basis_0[:] = surface.basis
    model.apply_stacking()
    poisson_results.append(evaluate_model(model))

print(f"evaluated {len(poisson_results)} Poisson roughness models")

In [ ]:
cmap = mpl.colormaps["turbo"]
density_offset = 15.0
stack_factor = 10.0


def plot_roughness_family(results, values, title, colorbar_label):
    """Plot density plus matched DWBA/kinematical rod families."""
    norm = mpl.colors.Normalize(vmin=values.min(), vmax=values.max())
    figure, axes = plt.subplots(
        1, 3, figsize=(15.0, 5.4), constrained_layout=True
    )
    for plot_index, (value, result) in enumerate(zip(values, results)):
        color = cmap(norm(value))
        scale = stack_factor**plot_index
        axes[0].plot(
            z, result["density"] + density_offset * plot_index,
            color=color, linewidth=1.2,
        )
        axes[1].semilogy(
            L_specular,
            np.abs(result["F_kinematic_specular"]) * scale,
            "--", color=color, linewidth=1.0,
        )
        axes[1].semilogy(
            L_specular, np.abs(result["F_dwba_specular"]) * scale,
            color=color, linewidth=1.25,
        )
        axes[2].semilogy(
            L_rod, np.abs(result["F_kinematic_rod"]) * scale,
            "--", color=color, linewidth=1.0,
        )
        axes[2].semilogy(
            L_rod, np.abs(result["F_dwba_rod"]) * scale,
            color=color, linewidth=1.25,
        )

    axes[0].set_xlabel(r"$z$ / Angstrom")
    axes[0].set_ylabel(r"stacked $|\rho_{00}(z)|$")
    axes[0].set_title("corresponding z-density")
    axes[1].set_title("specular (0, 0, L)")
    axes[2].set_title(
        rf"non-specular (0, 1, L), $\alpha_i=1.5\alpha_c$"
    )
    for axis in axes[1:]:
        axis.set_xlabel(r"$L$ / r.l.u.")
        axis.set_ylabel(r"stacked $|F|$ / electrons")
    for axis in axes:
        axis.grid(alpha=0.18, which="both")

    axes[1].legend(
        handles=[
            Line2D(
                [0], [0], color="0.25",
                label="DWBA (total on specular)",
            ),
            Line2D(
                [0], [0], color="0.25", linestyle="--",
                label="kinematical",
            ),
        ],
        loc="lower left",
    )
    colorbar = figure.colorbar(
        mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
        ax=axes, pad=0.015, shrink=0.86,
    )
    colorbar.set_label(colorbar_label)
    figure.suptitle(title)
    return figure


poisson_figure = plot_roughness_family(
    poisson_results, dissolved_layers,
    "Poisson dissolution: DWBA versus kinematical CTRs",
    "mean dissolved layers",
)
poisson_figure

## Deterministic layer-by-layer dissolution

The second construction replaces the Poisson correction by an explicit six-layer `UnitCell`. The lower eleven RuO2 layers remain a `Film`; atom occupancies in the surface cell encode removal of the upper six layers. This directly exercises the unit-cell contribution path in atomic DWBA.

In [ ]:
layerwise_removed = np.array([0.0, 0.25, 0.5, 0.75, 1.0, 3.0, 6.0])


def make_layerwise_dissolution_model(template, removed_layers):
    """Replace the Poisson correction with six explicit layers."""
    bulk = copy.deepcopy(template["bulk"])
    interface = copy.deepcopy(template["TiO2toRuO2"])
    film = copy.deepcopy(template["RuO2"])

    # The explicit surface cell supplies the upper six of the original
    # seventeen structural layers.
    film.basis[:] = 11.0
    film.basis_0[:] = film.basis

    lower_stack = CTRcalc.SXRDCrystal(
        bulk, film, interface, stacking=np.array([2, 1]),
        atten=template.atten,
    )
    lower_stack.apply_stacking()

    source_cell = film.unitcell
    source_layers = source_cell.split_in_layers()
    source_cycle = list(film.layer_order)
    next_index = (
        source_cycle.index(film.end_layer_number) + 1
    ) % len(source_cycle)
    surface_cycle = source_cycle[next_index:] + source_cycle[:next_index]

    surface_cell = CTRcalc.UnitCell(
        [source_cell.a[0], source_cell.a[1], 3.0 * source_cell.a[2]],
        np.rad2deg(source_cell.alpha),
        name="RuO2_layerwise_surface",
        layer_cycle=tuple(range(1, 7)),
        layer_behavior="ignore",
    )

    remaining_layers = 6.0 - removed_layers
    layer_occupancies = np.clip(
        remaining_layers - np.arange(6), 0.0, 1.0
    )
    for layer_index in range(6):
        source_layer_id = surface_cycle[layer_index % len(surface_cycle)]
        source_layer = source_layers[source_layer_id]
        source_origin = source_cell.layerpos[source_layer_id]
        target_origin = layer_index / 2.0
        occupancy = layer_occupancies[layer_index]

        for name, atom in zip(source_layer.names, source_layer.basis):
            surface_cell.addAtom(
                name,
                [
                    atom[1], atom[2],
                    (atom[3] - source_origin + target_origin) / 3.0,
                ],
                atom[4], atom[5], atom[6] * occupancy,
                layer=layer_index + 1,
            )
        surface_cell.layerpos[float(layer_index + 1)] = target_origin / 3.0

    surface_cell.setEnergy(source_cell._E)
    model = CTRcalc.SXRDCrystal(
        bulk, surface_cell, film, interface,
        stacking=np.array([3, 2, 1]), atten=template.atten,
    )
    model.apply_stacking()
    return model


layerwise_results = []
for removed in layerwise_removed:
    layerwise_model = make_layerwise_dissolution_model(
        model_template, removed
    )
    layerwise_results.append(evaluate_model(layerwise_model))

# A fully occupied six-layer cell reconstructs the pristine film. The
# small tolerance covers the changed floating-point summation order.
np.testing.assert_allclose(
    layerwise_results[0]["F_kinematic_specular"],
    poisson_results[0]["F_kinematic_specular"],
    rtol=2e-4, atol=1e-2,
)
np.testing.assert_allclose(
    layerwise_results[0]["density"], poisson_results[0]["density"],
    rtol=5e-4, atol=2e-4,
)
print(f"evaluated {len(layerwise_results)} explicit-layer models")

In [ ]:
layerwise_figure = plot_roughness_family(
    layerwise_results, layerwise_removed,
    "Layer-by-layer dissolution: DWBA versus kinematical CTRs",
    "layers removed",
)
layerwise_figure

The non-specular reference amplitude is identically zero, so differences there arise only from the four distorted-wave channels and their complex optical fields. On the specular rod, the piecewise-constant optical reference is subtracted from the atomic amplitude before the correction is combined with the Fresnel field. The plots express that total field in electron-equivalent units for comparison with the kinematical structure factor; the stored `reflectivity` arrays contain the same result directly as the physical dimensionless intensity.